# 16 · Phases 11 & 13 — RL-guided adaptive fusion networks (training order steps 5–7)

Every recording is processed with the configuration its context's policy selects:

```
PCG x(t) ─► descriptors ─► PCA ─► context k ─► policy π(k) = (wavelet, level, band, window, fusion)
                                                   │
           ┌───────────────────────────────────────┼──────────────────────────────┐
           ▼                                       ▼                              ▼
  T: raw PCG window ─► multi-scale 1-D CNN   S: STFT (band rows) ─► 2-D CNN   W: wavelet packet (wavelet, level, band) ─► 2-D CNN
           │  temporal attention pooling           │  pyramid stem + SE + TAP     │  pyramid stem + SE + TAP
           ▼                                       ▼                              ▼
          F_T ───────────────── α = softmax over enabled branches ─────────────── F_W
                             F = α_T F_T + α_S F_S + α_W F_W  ─►  MLP  ─►  p(murmur)
```

* **R1 · RL wavelet-only** — wavelet branch only, wavelet chosen per context by the bandit (Notebook 09)
* **R2 · Full RL, frozen policy** (Phase 13a) — the selected tabular agent's policy (Notebook 14)
* **R3 · Supernet fine-tuned under the frozen policy** (Phase 13b) — the weight-sharing supernet of
  Notebook 07 (which already serves every configuration) fine-tuned at LR/5 on the policy's configurations
* **R4 · Joint refinement** (Phase 13c) — the policy is refined with rewards computed from the
  *deep* model on validation recordings, with a policy step size β = 0.02 (5× smaller than the
  agent's α = 0.1), then the network is fine-tuned on the training split under the refined policy.

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import torch
import zlib

device = torch.device("cuda")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
seeds = CFG["deep"]["seeds"]
quiet = lambda s: print(s) if s.startswith("[") else None

wpol = load_json(P["results"] / "policies" / "rl_wavelet_only.json")
fpol = load_json(P["results"] / "policies" / "rl_full.json")
K = fpol["K"]
print("wavelet-only policy:", wpol["wavelets"])
print("full policy:", fpol["labels"])

In [ ]:
cfg_w = policy_configs(meta, space, np.array(wpol["policy_config"]), wpol["K"])
rows = run_experiment("R1_rl_wavelet_only", meta, signals, cfg_w, CFG, P, device, seeds=seeds, branches=("W",), log=quiet)
cfg_full = policy_configs(meta, space, np.array(fpol["policy"]), K)
rows += run_experiment("R2_full_rl_frozen_policy", meta, signals, cfg_full, CFG, P, device, seeds=seeds, log=quiet)
rows += run_experiment("R3_full_rl_finetuned", meta, signals, cfg_full, CFG, P, device, seeds=seeds, log=quiet,
                       init_from="supernet", lr=CFG["deep"]["lr"] / 5, epochs=15)

## Phase 13c — joint refinement of the policy with deep-model rewards

In [ ]:
prob = RLProblem(CFG, P, K=K)
mdp = prob.mdp("train")
va = np.where(meta.split == "val")[0]
fs = CFG["preprocessing"]["fs"]
noisy_signals = [add_noise(s, CFG["reward"]["robustness_snr_db"], np.random.default_rng(zlib.crc32(r.encode())), CFG["reward"]["robustness_noise"], fs)
                 for s, r in zip(signals, meta.record_id)]
ctx_all = meta[f"ctx_K{K}"].values
BETA, TOPN = 0.02, 3
y = meta.y.values
prior1 = y[meta.split == "train"].mean()


def deep_probs(model, cands, sigs, rec_idx):
    out = {}
    for c in cands:
        store = RecordStore([sigs[i] for i in rec_idx], [config_dict(space, c)] * len(rec_idx), fs, CFG["deep"])
        out[c] = predict(model, store, np.arange(len(rec_idx)), device)
    return out


refined_policies = {}
for seed in seeds:
    model = load_model("R3_full_rl_finetuned", seed, CFG, P).to(device)
    policy = np.array(fpol["policy"])
    new_policy = policy.copy()
    log_rows = []
    for k in range(K):
        top = list(np.argsort(-mdp.Rbar[k])[:TOPN])
        cands = list(dict.fromkeys([int(policy[k]), space.baseline] + [int(c) for c in top]))
        members = np.where(ctx_all[va] == k)[0]
        if len(members) < 5:
            continue
        rec = va[members]
        pc = deep_probs(model, cands, signals, rec)
        pn = deep_probs(model, cands, noisy_signals, rec)
        comp = reward_components(np.stack([pc[c] for c in cands]), np.stack([pn[c] for c in cands]), y[rec], prior1)
        Rdeep = reward_table(comp, prob.complexity[cands], CFG["reward"]["weights"], cands.index(space.baseline))
        Q = np.array([mdp.Rbar[k, c] for c in cands], float)
        for j in np.random.default_rng(seed).permutation(len(rec)):
            Q += BETA * (Rdeep[j] - Q)
        new_policy[k] = cands[int(np.argmax(Q))]
        log_rows.append({"seed": seed, "context": k, "old": space.label(int(policy[k])), "new": space.label(int(new_policy[k])),
                         "changed": bool(new_policy[k] != policy[k])})
    refined_policies[seed] = new_policy
    print(pd.DataFrame(log_rows).to_string(index=False))
    save_json({"K": K, "policy": [int(c) for c in new_policy], "labels": [space.label(int(c)) for c in new_policy]},
              P["results"] / "policies" / f"rl_joint_refined_s{seed}.json")

In [ ]:
for seed in seeds:
    cfg_ref = policy_configs(meta, space, refined_policies[seed], K)
    rows += run_experiment("R4_full_rl_joint_refined", meta, signals, cfg_ref, CFG, P, device, seeds=(seed,), log=quiet,
                           init_from="R3_full_rl_finetuned", lr=CFG["deep"]["lr"] / 10, epochs=8)
res = pd.DataFrame(rows)
res.to_csv(P["tables"] / "16_rl_models_runs.csv", index=False)
val_auc = res.groupby("model").val_auroc.mean()
proposed = val_auc[[m for m in val_auc.index if m.startswith(("R2", "R3", "R4"))]].idxmax()
save_json({"proposed_model": proposed, "criterion": "mean validation AUROC over seeds", "val_auroc": val_auc.to_dict()},
          P["results"] / "proposed_model.json")
print("proposed (selected on validation):", proposed)
mean_std_table([{"model": r["model"], **{k[5:]: v for k, v in r.items() if k.startswith("test_")}} for r in rows])